In [ ]:
import os
import re

# Takes the filename as input and parses out the User ID, Image Number, SessionNo/EyeSide
def parse_filename(filename, database="CASIAV1"):
    base = os.path.basename(filename)
    name, extension = os.path.splitext(base)
    
    # Initialize all optional fields
    user_id = "none"
    session_number = "none"
    eye_side = "none"
    image_number = "none"

    if database.upper() == "CASIAV1":
        # CASIA V1 format: {3_digit_user_id}_{Session_id}_{image_num}.jpg
        # Example: 001_1_01.jpg
        match = re.match(r"(\d{3})_(\d)_(\d+)", name)
        
        if not match:
            raise ValueError(f"Invalid CASIAV1 filename format: {filename}")

        user_id, session_number, image_number = match.groups()
        eye_side = "none" 

    elif database.upper() == "CASIA-IRIS-INTERVAL":
        # CASIA-Iris-Interval format: S1{3_digit_user_id}{eye_side}{image_no}.jpg
        # Example: S1001L01.jpg
        match = re.match(r"S1(\d{3})([LR])(\d{2})", name, re.IGNORECASE)
        
        if not match:
            raise ValueError(f"Invalid CASIA-Iris-Interval filename format: {filename}")

        user_id, eye_side, image_number = match.groups()
        session_number = "1"

    elif database.upper() == "IITD":
        # IITD format: {image_no}_{eyeside}.bmp (Filename only)
        # Directory structure: ./IITD/{3_digit_user_id}/...
        # Example Filename: 1_L.bmp or 10_R.bmp
        # Capture: Image Number (1 or 2 digits) and Eye Side (1 letter L/R)
        match = re.match(r"(\d{1,2})_([LR])", name, re.IGNORECASE)
        
        if not match:
            raise ValueError(f"Invalid IITD filename format: {filename}")
        
        image_number, eye_side = match.groups()
        # User ID and Session Number must be extracted from the path in scan_files
        session_number = "1" # Assuming a single session for initial processing

    else:
        raise ValueError(f"Unknown database specified: {database}")

    return {
        "user_id": user_id,
        "session_number": session_number,
        "image_number": image_number,
        "eye_side": eye_side.upper()
    }

# Takes the directory path as input
# Returns a dictionary of All image filepaths, and file metadata
def scan_files(path="./CASIA1", database="CASIAV1"):
    allfiles = []
    user_ids = set()
    database_upper = database.upper()

    for root, dirs, files in os.walk(path):
        # NEW LOGIC: Extract USER ID from directory for IITD
        current_user_id = "none"
        if database_upper == "IITD":
            # Assuming the path structure is like .../IITD/{user_id}/
            # The last directory name is the user_id (e.g., '001', '002', etc.)
            # We look for the directory name that consists of digits
            
            # os.path.split(root) gives (parent_path, directory_name)
            # This is safer than splitting the whole path string
            directory_name = os.path.basename(root) 
            if directory_name.isdigit() and len(directory_name) <= 3: # Check if it looks like a user_id
                current_user_id = directory_name
        
        for f in files:
            if f.lower().endswith(('.jpg', '.png', '.jpeg', '.bmp')):
                full_path = os.path.join(root, f)
                try:
                    meta = parse_filename(f, database) 
                    
                    # Insert the User ID extracted from the path for IITD
                    if database_upper == "IITD" and current_user_id != "none":
                         meta["user_id"] = current_user_id

                    # If CASIAV1, the user_id is in the filename, which is already in meta
                    # But if we are in the root directory (path), current_user_id will be 'none'
                    # We ensure we have a valid user ID before proceeding
                    if meta["user_id"] == "none":
                         continue # Skip files in directories that don't match the user_id pattern
                         
                    meta["filepath"] = full_path
                    allfiles.append(meta)
                    user_ids.add(meta["user_id"])
                except ValueError as e:
                    print(f"Skipping {full_path} (File: {f}): {e}") 
    return allfiles, sorted(list(user_ids))

file_metadata_casia_v1 = parse_filename("CASIA1/1/001_1_1.jpg", database="CASIAV1")
print(file_metadata_casia_v1)

file_metadata_casia_interval = parse_filename("CASIA-Iris-Interval/001/L/S1001L01.jpg", database="CASIA-IRIS-INTERVAL")
print(file_metadata_casia_interval)

file_metadata_iitd = parse_filename("IITD/205/01_R.bmp", database="IITD")
print(file_metadata_iitd)

In [ ]:
allfiles, user_ids = scan_files("./IITD", database="IITD")
print(allfiles)
print(user_ids)

In [ ]:
from tqdm import tqdm
import numpy as np
# function that runs the whole pipeline
from tqdm import tqdm
import numpy as np
import os # Make sure os is imported, which it is implicitly in the context
# function that runs the whole pipeline
def pipeline(dataset_path = "./CASIA1", save_visuals = False, save_intermediates = True):

    database_name = "CASIAV1"
    if "CASIA-Iris-Interval" in dataset_path:
        database_name = "CASIA-IRIS-INTERVAL"
    elif "IITD" in dataset_path:
        database_name = "IITD"
        
    ## Main Pipeline to run the model on all images
    print(f"Scanning Dataset {dataset_path} (Database: {database_name})")
    # CRITICAL: scan_files now requires the database_name argument
    files, user_ids = scan_files(dataset_path, database=database_name)
    print(f"Found {len(files)} files for {len(user_ids)} unique users")

    

    # init debug env
    print("Initialising Iris Pipeline")
    iris_pipeline = iris.IRISPipeline(env=iris.IRISPipeline.DEBUGGING_ENVIRONMENT)

    # init visualizer
    print("Initialising Iris Visualizer")
    iris_visualizer = iris.visualisation.IRISVisualizer()

    # Step 3: Prepare output dirs (Optional but good practice to use unique output dir)
    # Using the dataset_path in the output name might lead to long paths.
    # We will stick to the user's current approach of f"{dataset_path}_", 
    # but highly recommend using the database_name instead for a cleaner structure.
    output_vis_dir = os.path.join(f"{dataset_path}_", "outputs_vis")
    seg_dir = os.path.join(output_vis_dir, "segmentation")
    norm_dir = os.path.join(output_vis_dir, "normalized")
    code_dir = os.path.join(output_vis_dir, "codes")

    output_npz_dir = os.path.join(f"{dataset_path}_", "outputs_npz")
    temp_dir = os.path.join(output_npz_dir, "templates")
    seg_res_dir = os.path.join(output_npz_dir, "segmentation")
    norm_res_dir = os.path.join(output_npz_dir, "normalized") # This is the important info
    
    # Ensure folders exists (assuming ensure_dir is defined elsewhere)
    if save_visuals:
        ensure_dir(seg_dir)
        ensure_dir(norm_dir)
        ensure_dir(code_dir)

    if save_intermediates:
        ensure_dir(temp_dir)
        ensure_dir(seg_res_dir)
        ensure_dir(norm_res_dir)

    print(f"Fields: {files[0].keys()}")
    for file_meta in tqdm(files, desc="Processing iris images"):
        img_path = file_meta["filepath"]
        
        # 1. NEW LOGIC: Create a unique name from extracted metadata
        # This is essential for IITD and Interval since the base filename lacks User ID
        name_parts = [file_meta["user_id"], file_meta["session_number"], file_meta["eye_side"], file_meta["image_number"]]
        unique_name = "_".join([p for p in name_parts if p not in ["none", "1"]]) # Skip '1' and 'none' for brevity, but keep user_id, eye_side, image_number

        try:
            image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if image is None:
                raise IOError(f"Could not read Image {img_path}")
                
            # 2. CRITICAL UPDATE: Use file_meta['eye_side'] and 'unique_name'
            # The 'eye_side' is crucial for correct iris processing and normalization.
            current_eye_side = file_meta.get("eye_side", "right").lower()
            if current_eye_side == "l":
                current_eye_side = "left"
            else:
                current_eye_side = "right"
            
            iris_image = iris.IRImage(
                img_data=image, 
                image_id=unique_name, 
                eye_side=current_eye_side
            )
            output = iris_pipeline(iris_image)
            
            # 3. CRITICAL UPDATE: Use unique_name for saving results
            seg_path = os.path.join(seg_dir, f"{unique_name}_segmentation.jpg")
            norm_path = os.path.join(norm_dir, f"{unique_name}_normalized.jpg")
            temp_path = os.path.join(temp_dir, f"{unique_name}_template.npz")
            code_path = os.path.join(code_dir, f"{unique_name}_code.jpg")

            # Save intermediate arrays (if requested)
            if save_intermediates:
                try:
                    # segmentation map might be under call_trace['segmentation']
                    seg_arr = iris_pipeline.call_trace.get('segmentation', None)
                    if seg_arr is not None:
                        seg_npz_path = os.path.join(seg_res_dir, f"{unique_name}_segmentation.npz")
                        np.savez_compressed(seg_npz_path, segmentation=seg_arr)

                    norm_arr = iris_pipeline.call_trace.get('normalization', None)
                    if norm_arr is not None:
                        norm_npz_path = os.path.join(norm_res_dir, f"{unique_name}_normalized.npz")
                        np.savez_compressed(norm_npz_path, normalization=norm_arr)
                except Exception as e:
                    print(f"Warning: failed to save intermediates for {img_path}: {e}")

            if save_visuals:
                # Save segmentation result
                canvas = iris_visualizer.plot_segmentation_map(
                    # Use current_eye_side for plotting as well
                    ir_image=iris.IRImage(img_data=image, eye_side=current_eye_side),
                    segmap=iris_pipeline.call_trace['segmentation'],
                )
                plt.savefig(seg_path, bbox_inches='tight')
                plt.close('all')
                # Save normalization result
                canvas = iris_visualizer.plot_normalized_iris(
                    normalized_iris=iris_pipeline.call_trace['normalization'],
                    )
                plt.savefig(norm_path, bbox_inches='tight')
                plt.close('all')

                # Save printed Templates
                canvas = iris_visualizer.plot_iris_template(
                    iris_template=iris_pipeline.call_trace['encoder'],
                )
                plt.savefig(code_path, bbox_inches='tight')
                plt.close('all')

                # Save templates as npz
                iris_code = output['iris_template'].iris_codes[0]
                mask_code = output['iris_template'].iris_codes[1]
                np.savez_compressed(temp_path, iris_code = iris_code, mask_code = mask_code)
        except Exception as e:
            print(f"Error processing {img_path}: {e}")

In [2]:
!pip install tqdm


  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
Using cached tqdm-4.67.1-py3-none-any.whl (78 kB)


In [4]:
!pip install matplotlib


  Using cached contourpy-1.3.3-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.4.9-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (6.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 36.2 MB/s eta 0:00:00a 0:00:01
Using cached contourpy-1.3.3-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (362 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 45.2 MB/s eta 0:00:00
Using cached kiwisolver-1.4.9-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (1.5 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [matplotlib]7 [matplotlib]


In [17]:
import os
import re
from tqdm import tqdm
import numpy as np
import cv2
import matplotlib.pyplot as plt
import iris
# Note: ensure_dir, iris, and other modules are assumed to be available in your environment.

# Takes the filename as input and parses out the User ID, Image Number, SessionNo/EyeSide
def parse_filename(filename, database="CASIAV1"):
    base = os.path.basename(filename)
    name, extension = os.path.splitext(base)
    
    # Initialize all optional fields
    user_id = "none"
    session_number = "none"
    eye_side = "none"
    image_number = "none"

    db = database.upper()

    if db == "CASIAV1":
        # CASIA V1 format: {3_digit_user_id}_{Session_id}_{image_num}.jpg
        # Example: 001_1_01.jpg
        match = re.match(r"(\d{3})_(\d)_(\d+)", name)
        
        if not match:
            raise ValueError(f"Invalid CASIAV1 filename format: {filename}")

        user_id, session_number, image_number = match.groups()
        eye_side = "none"

    elif db in ("CASIA-IRIS-INTERVAL", "CASIA_IRIS_INTERVAL"):
        # CASIA-Iris-Interval format: S1{3_digit_user_id}{eye_side}{image_no}.jpg
        # Example: S1001L01.jpg  (S1 + 001 + L + 01)
        match = re.match(r"S1(\d{3})([LR])(\d{2})", name, re.IGNORECASE)
        
        if not match:
            raise ValueError(f"Invalid CASIA-Iris-Interval filename format: {filename}")

        user_id, eye_side, image_number = match.groups()
        session_number = "1"

    elif db in ("CASIA-THOUSAND", "CASIA_THOUSAND", "CASIA IRIS THOUSAND", "CASIA-IRIS-THOUSAND"):
        # CASIA Iris Thousand format (example path): .../002/R/S5002R00.jpg
        # Filename pattern: S5{3_digit_user_id}{eye_side}{image_no}.jpg
        # Example: S5002R00.jpg -> S5 + 002 + R + 00
        match = re.match(r"S5(\d{3})([LR])(\d{2})", name, re.IGNORECASE)
        if not match:
            raise ValueError(f"Invalid CASIA-Thousand filename format: {filename}")
        user_id, eye_side, image_number = match.groups()
        session_number = "1"  # CASIA Thousand doesn't use session ids in filename

    elif db == "IITD":
        # IITD format: {image_no}_{eyeside}.bmp (Filename only)
        # Directory structure: ./IITD/{3_digit_user_id}/...
        # Example Filename: 1_L.bmp or 10_R.bmp
        match = re.match(r"(\d{1,2})_([LR])", name, re.IGNORECASE)
        
        if not match:
            raise ValueError(f"Invalid IITD filename format: {filename}")
        
        image_number, eye_side = match.groups()
        # User ID and Session Number must be extracted from the path in scan_files
        session_number = "1" # Assuming a single session for initial processing

    else:
        raise ValueError(f"Unknown database specified: {database}")

    return {
        "user_id": user_id,
        "session_number": session_number,
        "image_number": image_number,
        "eye_side": eye_side.upper()
    }

# Takes the directory path as input
# Returns a list of file metadata dicts and a sorted list of user_ids
def scan_files(path="./CASIA1", database="CASIAV1"):
    allfiles = []
    user_ids = set()
    database_upper = database.upper()

    for root, dirs, files in os.walk(path):
        # Path-based USER ID extraction for datasets that don't carry user id in filename
        current_user_id = "none"

        # For IITD: path looks like .../IITD/{user_id}/...
        if database_upper == "IITD":
            directory_name = os.path.basename(root) 
            if directory_name.isdigit() and len(directory_name) <= 3:
                current_user_id = directory_name

        # For CASIA Thousand: files are in .../{user_id}/{L_or_R}/
        # e.g. /.../002/R/S5002R00.jpg -> here root ends with '.../002/R' so parent dir is user_id
        if database_upper in ("CASIA-THOUSAND", "CASIA_THOUSAND", "CASIA IRIS THOUSAND", "CASIA-IRIS-THOUSAND"):
            parent_dir = os.path.basename(os.path.dirname(root))  # this should return '002' if root is .../002/R
            if parent_dir.isdigit() and len(parent_dir) == 3:
                current_user_id = parent_dir

        for f in files:
            if f.lower().endswith(('.jpg', '.png', '.jpeg', '.bmp')):
                full_path = os.path.join(root, f)
                try:
                    meta = parse_filename(f, database) 
                    
                    # Insert the User ID extracted from the path for IITD and CASIA-THOUSAND (path has correct user id)
                    if database_upper in ("IITD", "CASIA-THOUSAND", "CASIA_THOUSAND", "CASIA IRIS THOUSAND", "CASIA-IRIS-THOUSAND") and current_user_id != "none":
                        meta["user_id"] = current_user_id

                    # Ensure we have a valid user ID before proceeding
                    if meta["user_id"] == "none":
                        # Skip files in directories that don't match the user_id pattern
                        # This behavior preserves the original code's skip behavior
                        continue 
                        
                    meta["filepath"] = full_path
                    allfiles.append(meta)
                    user_ids.add(meta["user_id"])
                except ValueError as e:
                    print(f"Skipping {full_path} (File: {f}): {e}") 
    return allfiles, sorted(list(user_ids))


# Example tests for parse_filename with the new dataset added
if __name__ == "__main__":
    file_metadata_casia_v1 = parse_filename("001_1_1.jpg", database="CASIAV1")
    print("CASIA V1:", file_metadata_casia_v1)

    file_metadata_casia_interval = parse_filename("S1001L01.jpg", database="CASIA-IRIS-INTERVAL")
    print("CASIA Interval:", file_metadata_casia_interval)

    file_metadata_iitd = parse_filename("01_R.bmp", database="IITD")
    print("IITD:", file_metadata_iitd)

    # CASIA Thousand example filename (S5 + userid 002 + R + 00)
    file_metadata_casia_thousand = parse_filename("S5002R00.jpg", database="CASIA-THOUSAND")
    print("CASIA Thousand:", file_metadata_casia_thousand)


# -------------------------
# pipeline function (unchanged functionality, only small detection tweak)
# -------------------------
def pipeline(dataset_path = "./CASIA1", save_visuals = False, save_intermediates = True):

    # Determine database type from dataset_path
    database_name = "CASIAV1"
    ds_lower = dataset_path.lower()
    if "casia-iris-interval" in ds_lower or "casia_iris_interval" in ds_lower or "interval" in ds_lower:
        database_name = "CASIA-IRIS-INTERVAL"
    elif "iitd" in ds_lower:
        database_name = "IITD"
    elif "thousand" in ds_lower or "casia-thousand" in ds_lower or "casia_thousand" in ds_lower or "casia iris thousand" in ds_lower:
        database_name = "CASIA-THOUSAND"
    else:
        # default remains CASIAV1
        database_name = "CASIAV1"
        
    ## Main Pipeline to run the model on all images
    print(f"Scanning Dataset {dataset_path} (Database: {database_name})")
    # scan_files now requires the database_name argument
    files, user_ids = scan_files(dataset_path, database=database_name)
    print(f"Found {len(files)} files for {len(user_ids)} unique users")

    files = files[:2]
    print(f"⚡ Running pipeline for {len(files)} sample files only")

    if len(files) == 0:
        print("No files found - exiting pipeline.")
        return

    # init debug env
    print("Initialising Iris Pipeline")
    iris_pipeline = iris.IRISPipeline(env=iris.IRISPipeline.DEBUGGING_ENVIRONMENT)

    # init visualizer
    print("Initialising Iris Visualizer")
    iris_visualizer = iris.visualisation.IRISVisualizer()

    # Step 3: Prepare output dirs (Optional but good practice to use unique output dir)
    output_vis_dir = os.path.join(f"{dataset_path}_", "outputs_vis")
    seg_dir = os.path.join(output_vis_dir, "segmentation")
    norm_dir = os.path.join(output_vis_dir, "normalized")
    code_dir = os.path.join(output_vis_dir, "codes")

    output_npz_dir = os.path.join(f"{dataset_path}_", "outputs_npz")
    temp_dir = os.path.join(output_npz_dir, "templates")
    seg_res_dir = os.path.join(output_npz_dir, "segmentation")
    norm_res_dir = os.path.join(output_npz_dir, "normalized")
    
    # Ensure folders exists (assuming ensure_dir is defined elsewhere)
    if save_visuals:
        ensure_dir(seg_dir)
        ensure_dir(norm_dir)
        ensure_dir(code_dir)

    if save_intermediates:
        ensure_dir(temp_dir)
        ensure_dir(seg_res_dir)
        ensure_dir(norm_res_dir)

    print(f"Fields: {files[0].keys()}")
    for file_meta in tqdm(files, desc="Processing iris images"):
        img_path = file_meta["filepath"]
        
        # Create a unique name from extracted metadata
        name_parts = [file_meta.get("user_id", "none"), file_meta.get("session_number", "none"),
                      file_meta.get("eye_side", "none"), file_meta.get("image_number", "none")]
        unique_name = "_".join([p for p in name_parts if p not in ["none", "1"]])

        try:
            image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if image is None:
                raise IOError(f"Could not read Image {img_path}")
                
            # Use file_meta['eye_side'] and 'unique_name'
            current_eye_side = file_meta.get("eye_side", "R").lower()
            if current_eye_side == "l":
                current_eye_side = "left"
            else:
                current_eye_side = "right"
            
            iris_image = iris.IRImage(
                img_data=image, 
                image_id=unique_name, 
                eye_side=current_eye_side
            )
            output = iris_pipeline(iris_image)
            
            # Paths for saving
            seg_path = os.path.join(seg_dir, f"{unique_name}_segmentation.jpg")
            norm_path = os.path.join(norm_dir, f"{unique_name}_normalized.jpg")
            temp_path = os.path.join(temp_dir, f"{unique_name}_template.npz")
            code_path = os.path.join(code_dir, f"{unique_name}_code.jpg")

            # Save intermediate arrays (if requested)
            if save_intermediates:
                try:
                    seg_arr = iris_pipeline.call_trace.get('segmentation', None)
                    if seg_arr is not None:
                        seg_npz_path = os.path.join(seg_res_dir, f"{unique_name}_segmentation.npz")
                        np.savez_compressed(seg_npz_path, segmentation=seg_arr)

                    norm_arr = iris_pipeline.call_trace.get('normalization', None)
                    if norm_arr is not None:
                        norm_npz_path = os.path.join(norm_res_dir, f"{unique_name}_normalized.npz")
                        np.savez_compressed(norm_npz_path, normalization=norm_arr)
                except Exception as e:
                    print(f"Warning: failed to save intermediates for {img_path}: {e}")

            if save_visuals:
                # Save segmentation result
                canvas = iris_visualizer.plot_segmentation_map(
                    ir_image=iris.IRImage(img_data=image, eye_side=current_eye_side),
                    segmap=iris_pipeline.call_trace.get('segmentation', None),
                )
                plt.savefig(seg_path, bbox_inches='tight')
                plt.close('all')
                # Save normalization result
                canvas = iris_visualizer.plot_normalized_iris(
                    normalized_iris=iris_pipeline.call_trace.get('normalization', None),
                )
                plt.savefig(norm_path, bbox_inches='tight')
                plt.close('all')

                # Save printed Templates
                canvas = iris_visualizer.plot_iris_template(
                    iris_template=iris_pipeline.call_trace.get('encoder', None),
                )
                plt.savefig(code_path, bbox_inches='tight')
                plt.close('all')

                # Save templates as npz (if present in output)
                iris_code = output.get('iris_template', None)
                if iris_code is not None:
                    # adjust indexing depending on your iris_template structure
                    try:
                        iris_codes = iris_code.iris_codes
                        mask_code = iris_codes[1] if len(iris_codes) > 1 else None
                        iris_code_arr = iris_codes[0] if len(iris_codes) > 0 else None
                        np.savez_compressed(temp_path, iris_code = iris_code_arr, mask_code = mask_code)
                    except Exception:
                        # fallback: try direct keys if different structure
                        try:
                            np.savez_compressed(temp_path, iris_template=iris_code)
                        except Exception as e:
                            print(f"Warning: failed to save template npz for {img_path}: {e}")
        except Exception as e:
            print(f"Error processing {img_path}: {e}")


CASIA V1: {'user_id': '001', 'session_number': '1', 'image_number': '1', 'eye_side': 'NONE'}
CASIA Interval: {'user_id': '001', 'session_number': '1', 'image_number': '01', 'eye_side': 'L'}
IITD: {'user_id': 'none', 'session_number': '1', 'image_number': '01', 'eye_side': 'R'}
CASIA Thousand: {'user_id': '002', 'session_number': '1', 'image_number': '00', 'eye_side': 'R'}


In [8]:
!pip install onnx onnxruntime pydantic==1.10.16 huggingface-hub pyyaml
!pip install --no-deps open-iris
!pip install opencv-python

  Using cached ml_dtypes-0.5.3-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.9 kB)
  Using cached coloredlogs-15.0.1-py2.py3-none-any.whl.metadata (12 kB)
  Using cached flatbuffers-25.9.23-py2.py3-none-any.whl.metadata (875 bytes)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached filelock-3.20.0-py3-none-any.whl.metadata (2.1 kB)
  Using cached humanfriendly-10.0-py2.py3-none-any.whl.metadata (9.2 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 40.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 66.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 65.1 MB/s eta 0:00:00
Using cached ml_dtypes-0.5.3-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (4.9 MB)
Using cached coloredlogs-15.0.1-py2.py3-none-any.whl (46 kB)
Using cached humanfriendly-10.0-py2.py3-none-any.whl (86 kB)
Using

In [ ]:


print(iris.__version__)

1.9.7


In [18]:
def ensure_dir(path):
    if not os.path.exists(path):
        os.makedirs(path, exist_ok=True)

In [19]:
if __name__ == "__main__":
    # Change this to your actual CASIA Thousand folder. Expected structure:
    # DATASET_PATH/
    #   000/
    #     L/
    #       S50000L00.jpg ... S50000L09.jpg
    #     R/
    #       S50000R00.jpg ...
    #   001/
    #     L/
    #     R/
    #   ...
    DATASET_PATH = "/home/nishkal/sg/iris_indexing/CASIA-Iris-Thousand" \
    "" \
    "" \
    "" \
    "" \
    ""  # <<--- set this to your dataset root

    # Run the pipeline (set save_visuals/save_intermediates as needed)
    pipeline(dataset_path=DATASET_PATH, save_visuals=True, save_intermediates=True)

Scanning Dataset /home/nishkal/sg/iris_indexing/CASIA-Iris-Thousand (Database: CASIA-THOUSAND)
Found 20000 files for 1000 unique users
⚡ Running pipeline for 2 sample files only
Initialising Iris Pipeline
Initialising Iris Visualizer
Fields: dict_keys(['user_id', 'session_number', 'image_number', 'eye_side', 'filepath'])


Processing iris images: 100%|██████████| 2/2 [00:01<00:00,  1.59it/s]


In [2]:
import os
import re
from tqdm import tqdm
import numpy as np
import cv2
import matplotlib.pyplot as plt
import iris
# Note: ensure_dir, iris, and other modules are assumed to be available in your environment.

# Takes the filename as input and parses out the User ID, Image Number, SessionNo/EyeSide
def parse_filename(filename, database="CASIAV1"):
    base = os.path.basename(filename)
    name, extension = os.path.splitext(base)
    
    # Initialize all optional fields
    user_id = "none"
    session_number = "none"
    eye_side = "none"
    image_number = "none"

    db = database.upper()

    if db == "CASIAV1":
        # CASIA V1 format: {3_digit_user_id}_{Session_id}_{image_num}.jpg
        # Example: 001_1_01.jpg
        match = re.match(r"(\d{3})_(\d)_(\d+)", name)
        
        if not match:
            raise ValueError(f"Invalid CASIAV1 filename format: {filename}")

        user_id, session_number, image_number = match.groups()
        eye_side = "none"

    elif db in ("CASIA-IRIS-INTERVAL", "CASIA_IRIS_INTERVAL"):
        # CASIA-Iris-Interval format: S1{3_digit_user_id}{eye_side}{image_no}.jpg
        # Example: S1001L01.jpg  (S1 + 001 + L + 01)
        match = re.match(r"S1(\d{3})([LR])(\d{2})", name, re.IGNORECASE)
        
        if not match:
            raise ValueError(f"Invalid CASIA-Iris-Interval filename format: {filename}")

        user_id, eye_side, image_number = match.groups()
        session_number = "1"

    elif db in ("CASIA-THOUSAND", "CASIA_THOUSAND", "CASIA IRIS THOUSAND", "CASIA-IRIS-THOUSAND"):
        # CASIA Iris Thousand format (example path): .../002/R/S5002R00.jpg
        # Filename pattern: S5{3_digit_user_id}{eye_side}{image_no}.jpg
        # Example: S5002R00.jpg -> S5 + 002 + R + 00
        match = re.match(r"S5(\d{3})([LR])(\d{2})", name, re.IGNORECASE)
        if not match:
            raise ValueError(f"Invalid CASIA-Thousand filename format: {filename}")
        user_id, eye_side, image_number = match.groups()
        session_number = "1"  # CASIA Thousand doesn't use session ids in filename

    elif db in ("IRIS-LAMP", "IRIS_LAMP"):
        # Iris-Lamp format: S2{3_digit_user_id}{eye_side}{image_no}.jpg
        # Example: S2001L01.jpg -> S2 + 001 + L + 01
        match = re.match(r"S2(\d{3})([LR])(\d{2})", name, re.IGNORECASE)
        if not match:
            raise ValueError(f"Invalid Iris-Lamp filename format: {filename}")
        user_id, eye_side, image_number = match.groups()
        session_number = "1"  # Iris-Lamp doesn't use session ids in filename

    elif db == "IITD":
        # IITD format: {image_no}_{eyeside}.bmp (Filename only)
        # Directory structure: ./IITD/{3_digit_user_id}/...
        # Example Filename: 1_L.bmp or 10_R.bmp
        match = re.match(r"(\d{1,2})_([LR])", name, re.IGNORECASE)
        
        if not match:
            raise ValueError(f"Invalid IITD filename format: {filename}")
        
        image_number, eye_side = match.groups()
        # User ID and Session Number must be extracted from the path in scan_files
        session_number = "1" # Assuming a single session for initial processing

    else:
        raise ValueError(f"Unknown database specified: {database}")

    return {
        "user_id": user_id,
        "session_number": session_number,
        "image_number": image_number,
        "eye_side": eye_side.upper() # This returns 'L' or 'R'
    }

# Takes the directory path as input
# Returns a list of file metadata dicts and a sorted list of user_ids
def scan_files(path="./CASIA1", database="CASIAV1"):
    allfiles = []
    user_ids = set()
    database_upper = database.upper()

    for root, dirs, files in os.walk(path):
        # Path-based USER ID extraction for datasets that don't carry user id in filename
        current_user_id = "none"

        # For IITD: path looks like .../IITD/{user_id}/...
        if database_upper == "IITD":
            directory_name = os.path.basename(root) 
            if directory_name.isdigit() and len(directory_name) <= 3:
                current_user_id = directory_name

        # For CASIA Thousand / Iris-Lamp: files are in .../{user_id}/{L_or_R}/
        # e.g. /.../002/R/S5002R00.jpg -> here root ends with '.../002/R' so parent dir is user_id
        if database_upper in ("CASIA-THOUSAND", "CASIA_THOUSAND", "CASIA IRIS THOUSAND", "CASIA-IRIS-THOUSAND", "IRIS-LAMP", "IRIS_LAMP"):
            parent_dir = os.path.basename(os.path.dirname(root))  # this should return '002' if root is .../002/R
            if parent_dir.isdigit() and len(parent_dir) == 3:
                current_user_id = parent_dir

        for f in files:
            if f.lower().endswith(('.jpg', '.png', '.jpeg', '.bmp')):
                full_path = os.path.join(root, f)
                try:
                    meta = parse_filename(f, database) 
                    
                    # Insert the User ID extracted from the path for IITD, CASIA-THOUSAND, and IRIS-LAMP
                    if database_upper in ("IITD", "CASIA-THOUSAND", "CASIA_THOUSAND", "CASIA IRIS THOUSAND", "CASIA-IRIS-THOUSAND", "IRIS-LAMP", "IRIS_LAMP") and current_user_id != "none":
                        meta["user_id"] = current_user_id

                    # Ensure we have a valid user ID before proceeding
                    if meta["user_id"] == "none":
                        # Skip files in directories that don't match the user_id pattern
                        # This behavior preserves the original code's skip behavior
                        continue 
                        
                    meta["filepath"] = full_path
                    allfiles.append(meta)
                    user_ids.add(meta["user_id"])
                except ValueError as e:
                    print(f"Skipping {full_path} (File: {f}): {e}") 
    return allfiles, sorted(list(user_ids))


# Example tests for parse_filename with the new dataset added
if __name__ == "__main__":
    file_metadata_casia_v1 = parse_filename("001_1_1.jpg", database="CASIAV1")
    print("CASIA V1:", file_metadata_casia_v1)

    file_metadata_casia_interval = parse_filename("S1001L01.jpg", database="CASIA-IRIS-INTERVAL")
    print("CASIA Interval:", file_metadata_casia_interval)

    file_metadata_iitd = parse_filename("01_R.bmp", database="IITD")
    print("IITD:", file_metadata_iitd)

    # CASIA Thousand example filename (S5 + userid 002 + R + 00)
    file_metadata_casia_thousand = parse_filename("S5002R00.jpg", database="CASIA-THOUSAND")
    print("CASIA Thousand:", file_metadata_casia_thousand)

    # Iris-Lamp example filename (S2 + userid 001 + L + 01)
    file_metadata_iris_lamp = parse_filename("S2001L01.jpg", database="IRIS-LAMP")
    print("Iris-Lamp:", file_metadata_iris_lamp)


# -------------------------
# pipeline function (FIXED)
# -------------------------
def pipeline(dataset_path = "./CASIA1", save_visuals = False, save_intermediates = True):

    # Determine database type from dataset_path
    database_name = "CASIAV1"
    ds_lower = dataset_path.lower()
    
    if "casia-iris-interval" in ds_lower or "casia_iris_interval" in ds_lower or "interval" in ds_lower:
        database_name = "CASIA-IRIS-INTERVAL"
    elif "iitd" in ds_lower:
        database_name = "IITD"
    elif "thousand" in ds_lower or "casia-thousand" in ds_lower or "casia_thousand" in ds_lower or "casia iris thousand" in ds_lower:
        database_name = "CASIA-THOUSAND"
    elif "iris-lamp" in ds_lower or "iris_lamp" in ds_lower or "lamp" in ds_lower:
        database_name = "IRIS-LAMP"
    else:
        # default remains CASIAV1
        database_name = "CASIAV1"
        
    ## Main Pipeline to run the model on all images
    print(f"Scanning Dataset {dataset_path} (Database: {database_name})")
    # scan_files now requires the database_name argument
    files, user_ids = scan_files(dataset_path, database=database_name)
    print(f"Found {len(files)} files for {len(user_ids)} unique users")

    if len(files) == 0:
        print("No files found - exiting pipeline.")
        return

    # init debug env
    print("Initialising Iris Pipeline")
    iris_pipeline = iris.IRISPipeline(env=iris.IRISPipeline.DEBUGGING_ENVIRONMENT)

    # init visualizer
    print("Initialising Iris Visualizer")
    iris_visualizer = iris.visualisation.IRISVisualizer()

    # Step 3: Prepare output dirs (Optional but good practice to use unique output dir)
    output_vis_dir = os.path.join(f"{dataset_path}_", "outputs_vis")
    seg_dir = os.path.join(output_vis_dir, "segmentation")
    norm_dir = os.path.join(output_vis_dir, "normalized")
    code_dir = os.path.join(output_vis_dir, "codes")

    output_npz_dir = os.path.join(f"{dataset_path}_", "outputs_npz")
    temp_dir = os.path.join(output_npz_dir, "templates")
    seg_res_dir = os.path.join(output_npz_dir, "segmentation")
    norm_res_dir = os.path.join(output_npz_dir, "normalized")
    
    # Ensure folders exists (assuming ensure_dir is defined elsewhere)
    if save_visuals:
        ensure_dir(seg_dir)
        ensure_dir(norm_dir)
        ensure_dir(code_dir)

    if save_intermediates:
        ensure_dir(temp_dir)
        ensure_dir(seg_res_dir)
        ensure_dir(norm_res_dir)

    print(f"Fields: {files[0].keys()}")
    for file_meta in tqdm(files, desc="Processing iris images"):
        img_path = file_meta["filepath"]
        
        # Create a unique name from extracted metadata
        name_parts = [file_meta.get("user_id", "none"), file_meta.get("session_number", "none"),
                      file_meta.get("eye_side", "none"), file_meta.get("image_number", "none")]
        # Clean up the name, remove default/redundant parts
        unique_name = "_".join([p for p in name_parts if p not in ["none", "1", None] and p])

        try:
            image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if image is None:
                raise IOError(f"Could not read Image {img_path}")
                
            # Use file_meta['eye_side'] and 'unique_name'
            # The .get() provides 'R' as a default if eye_side is missing, then .lower() makes it 'r'
            current_eye_side = file_meta.get("eye_side", "R").lower()
            
            # *** THIS IS THE FIX ***
            # Convert 'l' to 'left' and 'r' to 'right'
            if current_eye_side == "l":
                current_eye_side = "left"
            else:
                # Corrected 'current_eys_side' to 'current_eye_side'
                current_eye_side = "right"
            
            iris_image = iris.IRImage(
                img_data=image, 
                image_id=unique_name, 
                eye_side=current_eye_side # This will now be 'left' or 'right'
            )
            output = iris_pipeline(iris_image)
            
            # Paths for saving
            seg_path = os.path.join(seg_dir, f"{unique_name}_segmentation.jpg")
            norm_path = os.path.join(norm_dir, f"{unique_name}_normalized.jpg")
            temp_path = os.path.join(temp_dir, f"{unique_name}_template.npz")
            code_path = os.path.join(code_dir, f"{unique_name}_code.jpg")

            # Save intermediate arrays (if requested)
            if save_intermediates:
                try:
                    seg_arr = iris_pipeline.call_trace.get('segmentation', None)
                    if seg_arr is not None:
                        seg_npz_path = os.path.join(seg_res_dir, f"{unique_name}_segmentation.npz")
                        np.savez_compressed(seg_npz_path, segmentation=seg_arr)

                    norm_arr = iris_pipeline.call_trace.get('normalization', None)
                    if norm_arr is not None:
                        norm_npz_path = os.path.join(norm_res_dir, f"{unique_name}_normalized.npz")
                        np.savez_compressed(norm_npz_path, normalization=norm_arr)
                except Exception as e:
                    print(f"Warning: failed to save intermediates for {img_path}: {e}")

            if save_visuals:
                # Save segmentation result
                canvas = iris_visualizer.plot_segmentation_map(
                    ir_image=iris.IRImage(img_data=image, eye_side=current_eye_side),
                    segmap=iris_pipeline.call_trace.get('segmentation', None),
                )
                plt.savefig(seg_path, bbox_inches='tight')
                plt.close('all')
                # Save normalization result
                canvas = iris_visualizer.plot_normalized_iris(
                    normalized_iris=iris_pipeline.call_trace.get('normalization', None),
                )
                plt.savefig(norm_path, bbox_inches='tight')
                plt.close('all')

                # Save printed Templates
                canvas = iris_visualizer.plot_iris_template(
                    iris_template=iris_pipeline.call_trace.get('encoder', None),
                )
                plt.savefig(code_path, bbox_inches='tight')
                plt.close('all')

            # Save templates as npz (if present in output AND if save_intermediates is on)
            if save_intermediates:
                iris_code = output.get('iris_template', None)
                if iris_code is not None:
                    # adjust indexing depending on your iris_template structure
                    try:
                        iris_codes = iris_code.iris_codes
                        mask_code = iris_codes[1] if len(iris_codes) > 1 else None
                        iris_code_arr = iris_codes[0] if len(iris_codes) > 0 else None
                        np.savez_compressed(temp_path, iris_code = iris_code_arr, mask_code = mask_code)
                    except Exception:
                        # fallback: try direct keys if different structure
                        try:
                            np.savez_compressed(temp_path, iris_template=iris_code)
                        except Exception as e:
                            print(f"Warning: failed to save template npz for {img_path}: {e}")
        except Exception as e:
            print(f"Error processing {img_path}: {e}")

def ensure_dir(path):
    if not os.path.exists(path):
        os.makedirs(path, exist_ok=True)


if __name__ == "__main__":
    # Change this to your actual dataset folder.
    # Expected structure for Iris-Lamp:
    # DATASET_PATH/
    #   001/
    #     L/
    #       S2001L01.jpg ... S2001L20.jpg
    #     R/
    #       S2001R01.jpg ...
    #   002/
    #     L/
    #     R/
    #   ...
    #   411/
    
    # Example path for CASIA-Thousand:
    DATASET_PATH = "/home/nishkal/sg/iris_indexing/CASIA-Iris-Thousand"
    
    # Example path for Iris-Lamp:
    # DATASET_PATH = "/home/nishkal/sg/iris_indexing/CASIA-Iris-Lamp" # <<--- SET THIS TO YOUR DATASET ROOT
    pipeline(dataset_path=DATASET_PATH, save_visuals=True, save_intermediates=True)
    # Run the pipeline (set save_visuals/save_intermediates as needed)
    # if DATASET_PATH == "/path/to/your/Iris-Lamp" or DATASET_PATH == "/home/nishkal/sg/iris_indexing/CASIA-Iris-Lamp":
    #      print(f"Running pipeline for: {DATASET_PATH}")
         
    # else:
    #     print("Please update DATASET_PATH in the main block to point to your dataset.")

CASIA V1: {'user_id': '001', 'session_number': '1', 'image_number': '1', 'eye_side': 'NONE'}
CASIA Interval: {'user_id': '001', 'session_number': '1', 'image_number': '01', 'eye_side': 'L'}
IITD: {'user_id': 'none', 'session_number': '1', 'image_number': '01', 'eye_side': 'R'}
CASIA Thousand: {'user_id': '002', 'session_number': '1', 'image_number': '00', 'eye_side': 'R'}
Iris-Lamp: {'user_id': '001', 'session_number': '1', 'image_number': '01', 'eye_side': 'L'}
Scanning Dataset /home/nishkal/sg/iris_indexing/CASIA-Iris-Thousand (Database: CASIA-THOUSAND)
Found 20000 files for 1000 unique users
Initialising Iris Pipeline
Initialising Iris Visualizer
Fields: dict_keys(['user_id', 'session_number', 'image_number', 'eye_side', 'filepath'])


Processing iris images:   0%|          | 23/20000 [00:15<3:19:56,  1.67it/s]

Error processing /home/nishkal/sg/iris_indexing/CASIA-Iris-Thousand/391/L/S5391L07.jpg: 'NoneType' object has no attribute 'normalized_image'


Processing iris images:   0%|          | 25/20000 [00:16<3:08:07,  1.77it/s]

Error processing /home/nishkal/sg/iris_indexing/CASIA-Iris-Thousand/391/L/S5391L09.jpg: 'NoneType' object has no attribute 'normalized_image'


...

Error processing /home/nishkal/sg/iris_indexing/CASIA-Iris-Thousand/474/R/S5474R06.jpg: 'NoneType' object has no attribute 'normalized_image'


Processing iris images:   5%|▍         | 917/20000 [10:41<3:08:01,  1.69it/s]

Error processing /home/nishkal/sg/iris_indexing/CASIA-Iris-Thousand/474/R/S5474R08.jpg: 'NoneType' object has no attribute 'normalized_image'


Processing iris images:   5%|▍         | 953/20000 [11:07<3:42:25,  1.43it/s]


KeyboardInterrupt: 